In [1]:
import torch
import torch.nn as nn

In [2]:
tf = nn.Transformer()

C:\Users\22714\.conda\envs\feihu_dl_conda\Lib\site-packages\torch\nn\modules\transformer.py:143: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


In [22]:
model = nn.Transformer(
    d_model=256,
    nhead=8,
    num_encoder_layers=2,
    num_decoder_layers=2,
    dim_feedforward=2048,
    batch_first=True,
)

## 前向传播

In [16]:
d_model = 256
S = 8
T = 6
N = 4

In [17]:
src_vocab_size = 1000
tgt_vocab_size = 1500

In [18]:
# 定义数据(源和目标id序列)
src_ids = torch.randint(src_vocab_size, size=(N, S))
tgt_ids = torch.randint(tgt_vocab_size, size=(N, T))

In [19]:
# 定义词嵌入层
src_embedding = nn.Embedding(src_vocab_size, d_model)
tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)

### 整体流程(简单实现)

In [20]:
# 1. 词嵌入
src = src_embedding(src_ids)
tgt = tgt_embedding(tgt_ids)
print(src.shape, tgt.shape)

torch.Size([4, 8, 256]) torch.Size([4, 6, 256])


In [23]:
# 2. Transformer前向传播
out_put = model(src, tgt)
print(out_put.shape)

torch.Size([4, 6, 256])


## 加入掩码

In [24]:
# 填充掩码
padding_id = 0
src_key_padding_mask = (src_ids == padding_id)
tgt_key_padding_mask = (tgt_ids == padding_id)
print(src_key_padding_mask, tgt_key_padding_mask)

tensor([[False, False, False, False, False, False, False, False],
        [False, False, False, False, False, False, False, False],
        [False, False, False, False, False, False, False, False],
        [False, False, False, False, False, False, False, False]]) tensor([[False, False, False, False, False, False],
        [False, False, False, False, False, False],
        [False, False, False, False, False, False],
        [False, False, False, False, False, False]])


In [25]:
memory_key_padding_mask = src_key_padding_mask

In [26]:
# 因果性掩码(自回归掩码)
tgt_mask = model.generate_square_subsequent_mask(T).bool()
print(tgt_mask)

tensor([[False,  True,  True,  True,  True,  True],
        [False, False,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True],
        [False, False, False, False,  True,  True],
        [False, False, False, False, False,  True],
        [False, False, False, False, False, False]])


In [27]:
out_put = model(
    src,
    tgt,
    src_key_padding_mask=src_key_padding_mask,
    tgt_key_padding_mask=tgt_key_padding_mask,
    memory_key_padding_mask=memory_key_padding_mask,
    tgt_mask=tgt_mask,
)
print(out_put.shape)

torch.Size([4, 6, 256])


## 编码

In [28]:
# 简单测试
memory = model.encoder(src)
print(memory.shape)

torch.Size([4, 8, 256])


In [29]:
# 加入掩码
memory = model.encoder(src, src_key_padding_mask=src_key_padding_mask)
print(memory.shape)

torch.Size([4, 8, 256])


## 解码

In [30]:
# 简单测试
out_put = model.decoder(tgt, memory)
print(out_put.shape)

torch.Size([4, 6, 256])


In [31]:
# 加入掩码
out_put = model.decoder(
    tgt,
    memory,
    memory_key_padding_mask=memory_key_padding_mask,
    tgt_key_padding_mask=tgt_key_padding_mask,
    tgt_mask=tgt_mask,
)
print(out_put.shape)

torch.Size([4, 6, 256])
